# Reality2Unity — Week 1 Baseline (Colab)

Google Colab에서 **런타임 → 런타임 유형 변경 → GPU**를 선택한 뒤 위에서부터 실행하세요.


In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/mir0173/Reality2Unity.git"
BRANCH = "develop"
REPO_DIR = Path("/content/Reality2Unity")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )

required = [
    "python/main.py",
    "python/depth_estimation/midas.py",
    "python/terrain_generation/legacy_converter.py",
]


## 실행 패키지 준비



In [ ]:
%pip -q install timm opencv-python-headless matplotlib


In [ ]:
%matplotlib inline
import sys

python_dir = str(REPO_DIR / "python")
if python_dir not in sys.path:
    sys.path.insert(0, python_dir)

from main import main


## 입력 이미지 업로드



In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

image_path = Path('/content/drive/MyDrive/Reality2Unity/test.png')

print("Image:", image_path)

## Baseline 실행



In [ ]:
main(str(image_path))


# Reality2Unity — Week 2 (Colab)

## 깊이맵 및 이미지

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

from depth_estimation.midas import load_midas_model, estimate_depth

midas, transform, device = load_midas_model()

depth = estimate_depth(str(image_path), midas, transform, device)

image = cv2.imread(str(image_path))
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

## 	초점거리 비교

In [ ]:

H, W = depth.shape

depth_norm = (depth - depth.min()) / (depth.max() - depth.min())
Z = 1 / (depth_norm + 0.15)

step = 2

u, v = np.meshgrid(
    np.arange(0, W, step),
    np.arange(0, H, step)
)

z = Z[::step, ::step]
rgb = image[::step, ::step] / 255.0
mask = depth_norm[::step, ::step] > 0.1

cx = (W - 1) / 2
cy = (H - 1) / 2

fig = plt.figure(figsize=(18, 6))

for i, scale in enumerate([0.5, 1.0, 2.0]):
    fx = W * scale
    fy = W * scale

    x = (u - cx) * z / fx
    y = -(v - cy) * z / fy

    ax = fig.add_subplot(1, 3, i + 1, projection='3d')

    ax.scatter(
        x[mask],
        y[mask],
        z[mask],
        c=rgb[mask],
        s=1
    )

    ax.view_init(elev=20, azim=-70)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-2, 2)
    ax.set_zlim(0, 7)
    ax.set_title(f'Focal Length = {scale}W')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')

    print(f'Focal Length: {scale}W')
    print(f'X range: {np.ptp(x[mask]):.3f}')
    print(f'Y range: {np.ptp(y[mask]):.3f}')

plt.tight_layout()
plt.show()


In [ ]:
fig = plt.figure(figsize=(18, 6))

fx = W
fy = W

for i, offset in enumerate([0.05, 0.15, 0.5]):
    Z = 1 / (depth_norm + offset)
    z = Z[::step, ::step]

    x = (u - cx) * z / fx
    y = -(v - cy) * z / fy

    ax = fig.add_subplot(1, 3, i + 1, projection='3d')
    ax.scatter(x[mask], y[mask], z[mask], c=rgb[mask], s=1)

    ax.view_init(elev=20, azim=-70)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-2, 2)
    ax.set_zlim(0, 7)

    ax.set_title(f'Depth Offset = {offset}')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')

    print(f'Offset: {offset}')
    print(f'Z range: {np.ptp(z[mask]):.3f}')

plt.tight_layout()
plt.show()

## Point Cloud 저장

In [ ]:

from google.colab import files

H, W = depth.shape

depth_norm = (depth - depth.min()) / (depth.max() - depth.min())
Z = 1 / (depth_norm + 0.15)

step = 2

u, v = np.meshgrid(
    np.arange(0, W, step),
    np.arange(0, H, step)
)

z = Z[::step, ::step]

fx = W
fy = W
cx = (W - 1) / 2
cy = (H - 1) / 2

x = (u - cx) * z / fx
y = -(v - cy) * z / fy

mask = depth_norm[::step, ::step] > 0.1

points = np.column_stack((x[mask], y[mask], z[mask]))
colors = image[::step, ::step][mask]

output_path = "/content/drive/MyDrive/Reality2Unity/pointcloud_baseline.ply"

with open(output_path, "w") as f:
    f.write("ply\n")
    f.write("format ascii 1.0\n")
    f.write(f"element vertex {len(points)}\n")
    f.write("property float x\n")
    f.write("property float y\n")
    f.write("property float z\n")
    f.write("property uchar red\n")
    f.write("property uchar green\n")
    f.write("property uchar blue\n")
    f.write("end_header\n")

    for p, c in zip(points, colors):
        f.write(f"{p[0]} {p[1]} {p[2]} {int(c[0])} {int(c[1])} {int(c[2])}\n")

print("Saved:", output_path)
print("Number of points:", len(points))


# Reality2Unity — Week 3 (Colab)

## 다중 이미지 복원


In [ ]:
from pathlib import Path

image_dir = Path("/content/drive/MyDrive/Reality2Unity/multiview/images")

images = sorted(p for p in image_dir.iterdir() if p.suffix.lower() in [".jpg", ".jpeg", ".png"])

print("Number of images:", len(images))

for p in images:
    print(p.name)


In [ ]:
!apt-get update -qq
!apt-get install -y colmap

!colmap -h

In [ ]:
import shutil
import subprocess
from pathlib import Path

workspace = Path("/content/reality2unity_multiview")
local_images = workspace / "images"
sparse_dir = workspace / "sparse"
database = workspace / "database.db"

if database.exists():
    raise FileExistsError("기존 COLMAP 실행 데이터가 있습니다. 먼저 결과를 확인하세요.")

local_images.mkdir(parents=True, exist_ok=True)
sparse_dir.mkdir(parents=True, exist_ok=True)

for p in images:
    shutil.copy2(p, local_images / p.name)

def colmap_help(command):
    result = subprocess.run(
        ["colmap", command, "-h"],
        capture_output=True,
        text=True
    )
    return result.stdout + result.stderr

extract_help = colmap_help("feature_extractor")
match_help = colmap_help("exhaustive_matcher")

extract_gpu_option = (
    "FeatureExtraction.use_gpu"
    if "FeatureExtraction.use_gpu" in extract_help
    else "SiftExtraction.use_gpu"
)

match_gpu_option = (
    "FeatureMatching.use_gpu"
    if "FeatureMatching.use_gpu" in match_help
    else "SiftMatching.use_gpu"
)

subprocess.run([
    "colmap", "feature_extractor",
    "--database_path", str(database),
    "--image_path", str(local_images),
    f"--{extract_gpu_option}", "0"
], check=True)

subprocess.run([
    "colmap", "exhaustive_matcher",
    "--database_path", str(database),
    f"--{match_gpu_option}", "0"
], check=True)

subprocess.run([
    "colmap", "mapper",
    "--database_path", str(database),
    "--image_path", str(local_images),
    "--output_path", str(sparse_dir)
], check=True)

print("Sparse models:", list(sparse_dir.iterdir()))

## 복원 결과 분석

In [ ]:
import subprocess

model_path = "/content/reality2unity_multiview/sparse/0"

result = subprocess.run(
    ["colmap", "model_analyzer", "--path", model_path],
    capture_output=True,
    text=True,
    check=True
)

print(result.stdout)
print(result.stderr)

In [ ]:
from pathlib import Path
import subprocess

model_path = "/content/reality2unity_multiview/sparse/0"
output_path = Path("/content/drive/MyDrive/Reality2Unity/sparse_multiview.ply")

subprocess.run([
    "colmap", "model_converter",
    "--input_path", model_path,
    "--output_path", str(output_path),
    "--output_type", "PLY"
], check=True)

print("Saved:", output_path)
print("File size:", output_path.stat().st_size, "bytes")

In [ ]:
from pathlib import Path
import shutil

source = Path("/content/reality2unity_multiview")
backup = Path("/content/drive/MyDrive/Reality2Unity/multiview/colmap_output")

shutil.copytree(
    source / "sparse",
    backup / "sparse",
    dirs_exist_ok=True
)

shutil.copy2(
    source / "database.db",
    backup / "database.db"
)

print("COLMAP backup completed:", backup)

## 이미지 왜곡 보정

In [ ]:
import subprocess
from pathlib import Path

workspace = Path("/content/reality2unity_multiview")
dense_dir = workspace / "dense"

subprocess.run([
    "colmap", "image_undistorter",
    "--image_path", str(workspace / "images"),
    "--input_path", str(workspace / "sparse/0"),
    "--output_path", str(dense_dir),
    "--output_type", "COLMAP",
    "--max_image_size", "1600"
], check=True)

print("Dense workspace:", dense_dir)
print("Images:", len(list((dense_dir / "images").iterdir())))

In [ ]:
%pip -q install pycolmap-cuda12

import pycolmap

print("PyCOLMAP version:", pycolmap.__version__)
print("PatchMatch available:", hasattr(pycolmap, "patch_match_stereo"))
!nvidia-smi

In [ ]:
import pycolmap

print("PyCOLMAP version:", pycolmap.__version__)
print("CUDA enabled:", pycolmap.has_cuda)

## Dense 깊이맵 생성

In [ ]:
import subprocess
import sys

script = """
import pycolmap

options = pycolmap.PatchMatchOptions()
options.max_image_size = 1200
options.cache_size = 2.0

pycolmap.patch_match_stereo(
    "/content/reality2unity_multiview/dense",
    options=options
)

print("PatchMatch completed")
"""

subprocess.run(
    [sys.executable, "-u", "-c", script],
    check=True
)

In [ ]:
from pathlib import Path

dense_dir = Path("/content/reality2unity_multiview/dense")
depth_dir = dense_dir / "stereo" / "depth_maps"

geometric = list(depth_dir.glob("*.geometric.bin"))
photometric = list(depth_dir.glob("*.photometric.bin"))

print("Geometric depth maps:", len(geometric))
print("Photometric depth maps:", len(photometric))

for p in sorted(geometric)[:5]:
    print(p.name, p.stat().st_size, "bytes")

In [ ]:
import subprocess
import sys
from pathlib import Path
import shutil

dense_dir = Path("/content/reality2unity_multiview/dense")
fused_path = dense_dir / "fused.ply"

script = """
import pycolmap

options = pycolmap.StereoFusionOptions()
options.max_image_size = 1200
options.cache_size = 2.0
options.use_cache = True

pycolmap.stereo_fusion(
    "/content/reality2unity_multiview/dense/fused.ply",
    "/content/reality2unity_multiview/dense",
    input_type="geometric",
    options=options,
    output_type="ply"
)
"""

subprocess.run(
    [sys.executable, "-u", "-c", script],
    check=True
)

if not fused_path.is_file():
    raise FileNotFoundError(fused_path)

drive_path = Path("/content/drive/MyDrive/Reality2Unity/dense_multiview.ply")
shutil.copy2(fused_path, drive_path)

print("Saved:", drive_path)
print("File size:", drive_path.stat().st_size, "bytes")

## Mesh 생성

In [ ]:
import subprocess
import shutil
from pathlib import Path

dense_dir = Path("/content/reality2unity_multiview/dense")

input_path = dense_dir / "fused.ply"
mesh_path = dense_dir / "meshed-poisson.ply"

subprocess.run([
    "colmap", "poisson_mesher",
    "--input_path", str(input_path),
    "--output_path", str(mesh_path)
], check=True)

if not mesh_path.is_file():
    raise FileNotFoundError(mesh_path)

drive_path = Path(
    "/content/drive/MyDrive/Reality2Unity/meshed_poisson.ply"
)

shutil.copy2(mesh_path, drive_path)

print("Saved:", drive_path)
print("File size:", drive_path.stat().st_size, "bytes")

## Trim 기준 값 변화 실험

In [ ]:
from pathlib import Path
import subprocess
import shutil

dense_dir = Path("/content/reality2unity_multiview/dense")
input_path = dense_dir / "fused.ply"

for trim in [5, 2]:
    mesh_path = dense_dir / f"meshed-poisson-trim{trim}.ply"

    subprocess.run([
        "colmap", "poisson_mesher",
        "--input_path", str(input_path),
        "--output_path", str(mesh_path),
        "--PoissonMeshing.trim", str(trim)
    ], check=True)

    drive_path = Path(
        f"/content/drive/MyDrive/Reality2Unity/meshed_poisson_trim{trim}.ply"
    )

    shutil.copy2(mesh_path, drive_path)

    print("Trim:", trim)
    print("Saved:", drive_path)
    print("Size:", drive_path.stat().st_size, "bytes")

## Mesh 생성 방식 변경

Delaunay Mesh 생성 실패

시도: COLMAP Delaunay Meshing을 이용한 3D Mesh 생성

현상: Integrating image 단계에서 진행되지 않음

추정 원인: 구버전 COLMAP의 문제 가능성 (정확한 원인 미확인)

후속 조치: Dense Reconstruction의 해상도 및 품질 개선을 우선 진행

# Reality2Unity — Week 4 (Colab)

## 고해상도 확인

In [ ]:
from pathlib import Path
from google.colab import drive
import shutil

drive.mount("/content/drive")

project_dir = Path("/content/drive/MyDrive/Reality2Unity")
workspace = Path("/content/reality2unity_multiview")

source_images = project_dir / "multiview/images"
local_images = workspace / "images"

sparse_dir = workspace / "sparse/0"
sparse_backup = project_dir / "multiview/colmap_output/sparse/0"

if not source_images.is_dir():
    raise FileNotFoundError(source_images)

local_images.mkdir(parents=True, exist_ok=True)

images = sorted(
    p for p in source_images.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
)

for image in images:
    destination = local_images / image.name
    if not destination.exists():
        shutil.copy2(image, destination)

if not sparse_dir.is_dir():
    if not sparse_backup.is_dir():
        raise FileNotFoundError(sparse_backup)

    sparse_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(sparse_backup, sparse_dir)

for filename in ["cameras.bin", "images.bin", "points3D.bin"]:
    if not (sparse_dir / filename).is_file():
        raise FileNotFoundError(sparse_dir / filename)

print("Images:", len(images))
print("Sparse model:", sparse_dir)
print("COLMAP installed:", shutil.which("colmap") is not None)

In [ ]:
import subprocess
from pathlib import Path

dense_dir = workspace / "dense_1600"

prepared_images = dense_dir / "images"

existing = (
    [p for p in prepared_images.iterdir() if p.is_file()]
    if prepared_images.is_dir()
    else []
)

if len(existing) != len(images):
    subprocess.run([
        "colmap", "image_undistorter",
        "--image_path", str(local_images),
        "--input_path", str(sparse_dir),
        "--output_path", str(dense_dir),
        "--output_type", "COLMAP",
        "--max_image_size", "1600"
    ], check=True)

prepared = [
    p for p in prepared_images.iterdir()
    if p.is_file()
]

print("Dense workspace:", dense_dir)
print("Images:", len(prepared))

In [ ]:
import subprocess
import sys
from pathlib import Path

script = f"""
import pycolmap

if not pycolmap.has_cuda:
    raise RuntimeError("CUDA is not available")

options = pycolmap.PatchMatchOptions()
options.max_image_size = 1600
options.cache_size = 2.0

pycolmap.patch_match_stereo(
    "{dense_dir}",
    options=options
)

print("PatchMatch 1600 completed", flush=True)
"""

process = subprocess.Popen(
    [sys.executable, "-u", "-c", script],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line.rstrip(), flush=True)

if process.wait() != 0:
    raise RuntimeError("PatchMatch 1600 failed")

depth_dir = dense_dir / "stereo/depth_maps"
depth_maps = list(depth_dir.glob("*.geometric.bin"))

print("Geometric depth maps:", len(depth_maps))

In [ ]:
import subprocess
import sys
import shutil
from pathlib import Path

fused_path = dense_dir / "fused.ply"

script = f"""
import pycolmap

options = pycolmap.StereoFusionOptions()
options.max_image_size = 1200
options.cache_size = 2.0
options.use_cache = True

pycolmap.stereo_fusion(
    "{fused_path}",
    "{dense_dir}",
    input_type="geometric",
    options=options,
    output_type="ply"
)

print("Stereo Fusion completed", flush=True)
"""

subprocess.run(
    [sys.executable, "-u", "-c", script],
    check=True
)

if not fused_path.is_file():
    raise FileNotFoundError(fused_path)

output_path = project_dir / "dense_multiview_1600.ply"
shutil.copy2(fused_path, output_path)

print("Saved:", output_path)
print("File size:", output_path.stat().st_size, "bytes")

In [ ]:
def count_ply_vertices(path):
    with open(path, "rb") as f:
        for line in f:
            text = line.decode("ascii").strip()

            if text.startswith("element vertex"):
                return int(text.split()[-1])

            if text == "end_header":
                break

    raise ValueError("Vertex count not found")

baseline_path = project_dir / "dense_multiview.ply"
improved_path = project_dir / "dense_multiview_1600.ply"

baseline_points = count_ply_vertices(baseline_path)
improved_points = count_ply_vertices(improved_path)

print("Baseline (1200):", baseline_points)
print("Improved (1600):", improved_points)
print("Point ratio:", round(improved_points / baseline_points, 3))

## 저장 전 출력 정리

실행 후 Git에 반영하기 전에는 Colab의 **수정 → 모든 출력 지우기**를 실행하세요. Colab 설정에서 **저장 시 코드 셀 출력 생략**을 활성화하세요.